# Misinformation Models  — Opinion Classification
Driver notebook: runs each model, collects predictions, compares metrics.  
Add new models in the **Run models** cell. Train/dev only — test set not touched.

### Import configuration and packages ###

In [21]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import torch
import cnn_baseline as cnn
from config import DATA_DIR, FASTTEXT_PATH
from preprocess import preprocess
from metrics import compute_metrics, print_confusion_matrix, print_sklearn_report, error_analysis

DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()           else
    "cpu"
)
print(f"Device: {DEVICE}")

Device: mps


In [22]:
# Load data (shared across all models)
train_rows = cnn.load_csv(DATA_DIR / "mis_df_train.csv")
dev_rows   = cnn.load_csv(DATA_DIR / "mis_df_dev.csv")
print(f"Train: {len(train_rows)} | Dev: {len(dev_rows)}")

Train: 600 | Dev: 200


### Run models ###


In [17]:
# Run CNN model here
# Each model should produce (preds, labels, probs) and store them in `results`.
# Add new models below following the same pattern.
results = {}

# TextCNN
vocab        = cnn.build_vocab(train_rows, preprocess)
embed_matrix = cnn.load_fasttext_vectors(FASTTEXT_PATH, vocab)
train_loader = cnn.make_loader(train_rows, vocab, shuffle=True,  tokenize_fn=preprocess)
dev_loader   = cnn.make_loader(dev_rows,   vocab, shuffle=False, tokenize_fn=preprocess)
cnn_model    = cnn.TextCNN(len(vocab), embed_matrix).to(DEVICE)
cnn_model    = cnn.train_model(cnn_model, train_loader, dev_loader, train_rows, DEVICE)
results["TextCNN"] = cnn.predict(cnn_model, dev_loader, DEVICE)



Loading FastText vectors from /Users/jennifer/git/MDS-CL/cc.en.300.vec …
  3875/4091 vocab tokens found in FastText vectors (94.7%)
Epoch   1 | loss=0.7659 | dev_macro_f1=0.6795
Epoch   2 | loss=0.6457 | dev_macro_f1=0.6840
Epoch   3 | loss=0.5884 | dev_macro_f1=0.6928
Epoch   4 | loss=0.5423 | dev_macro_f1=0.7029
Epoch   5 | loss=0.5027 | dev_macro_f1=0.6748
Epoch   6 | loss=0.4421 | dev_macro_f1=0.7273
Epoch   7 | loss=0.3908 | dev_macro_f1=0.6799
Epoch   8 | loss=0.3538 | dev_macro_f1=0.7206
Epoch   9 | loss=0.2885 | dev_macro_f1=0.7070
Epoch  10 | loss=0.2653 | dev_macro_f1=0.7083
Epoch  11 | loss=0.2291 | dev_macro_f1=0.7134
  Early stopping (no improvement for 5 epochs). Best F1: 0.7273


In [18]:
# future models go here, e.g.:
# import logreg_baseline as lr
# results["LogReg"] = lr.run(train_rows, dev_rows)

In [ ]:
#Create table to compare models across metrics
rows = []
for name, (preds, labels, probs) in results.items():
    m = compute_metrics(preds, labels, probs)
    rows.append({"Model": name, "Accuracy": m["accuracy"], "Macro F1": m["macro_f1"],
                 "F1 (not-op)": m["f1_class0"], "F1 (opinion)": m["f1_class1"],
                 "AUC-ROC": m.get("auc_roc", float("nan"))})

pd.set_option("display.float_format", "{:.4f}".format)
pd.DataFrame(rows).set_index("Model")

,Accuracy,Macro F1,F1 (not-op),F1 (opinion),AUC-ROC
Model,,,,,
TextCNN,0.7300,0.7273,0.7545,0.7000,0.7844


In [20]:
#Model details
# 
for name, (preds, labels, probs) in results.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    print_confusion_matrix(preds, labels)
    print()
    print_sklearn_report(preds, labels)
    print()
    error_analysis(dev_rows, preds, labels)


TextCNN
Confusion matrix (rows=true, cols=predicted):
                 pred=0  pred=1
  true=0 (not-op):    83      34
  true=1 (opinion):   20      63

              precision    recall  f1-score   support

 not-opinion       0.81      0.71      0.75       117
     opinion       0.65      0.76      0.70        83

    accuracy                           0.73       200
   macro avg       0.73      0.73      0.73       200
weighted avg       0.74      0.73      0.73       200


False Positives (predicted opinion, actually not) — 5 shown:
  [17] 'Birds in a blizzard. We put out extra sunflower seeds since their usual food sources just got‚Ä¶ https://www.instagram.c'
  [25] "I'm getting used to seeing a layer of ash on my car each morning - and I'm in AB. We've  only seen a red sun this week. "
  [55] "@jihettly @esd2000 good morning! Yes I'm in the middle of the blizzard. I have plenty of food so I'm happy! Lol. Stay wa"
  [59] 'Re: Hurricane Matthew: All of the @ASUTennis Student-Athlet